# Train scale_strong seeds 0–11 + probe (Kaggle)

Trains all 12 **scale_strong** encoders (Shapes3D, zoom-only views, 0.6–1.4×) from scratch
to the 50-epoch cut, then probes the finished cell into its `stacks.npz`. Fresh
training starts from each seed's random init, so there is nothing to upload and
no `Add Input` step.

## One-time setup
1. **Settings → Accelerator → `GPU T4 x2`** (or single T4 — both work; x2 runs
   two seeds at once).
2. **Settings → Internet → On** (clone + install + Shapes3D download).

## Run order
Sections 1–4 top to bottom (clone, install, data), then **section 5** (train).
Section 5 is idempotent + resumable: it skips a seed once its `backbone.pt`
exists and resumes a partial seed from its own `last_ckpt.pt`, so **re-run it
after a session ends** to continue. **Save Version** every few hours so
`/kaggle/working` persists (Kaggle sessions cap ~12 h).

**New-condition check:** this is the first cell trained with the scale
augmentation, so once the first seeds finish (~5 h) run section 7 and look at
the collapse diagnostics — verify the 50-epoch cut is healthy before letting
the remaining seeds spend the quota (interrupt section 5 if it is not).

When all 12 encoders exist, run **section 8** to probe the cell into
`results/probes/scale_strong/`. The per-encoder quality gate runs inside the
probe step; gate-failed seeds are recorded in `meta.json` and excluded by the
statistics layer.

Wall-clock: ~5 GPU-h per full Shapes3D encoder; 12 seeds ÷ 2 GPUs ≈ **~30 h ≈
3 sessions** (~1 week at 30 quota-h/wk), plus **~3–4 h** for the probe step.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")

## 4. Download Shapes3D + build the image cache
scale_strong is a Shapes3D cell. `--build-cache` decompresses once into an
uncompressed memmap the trainers mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Train — scale_strong seeds 0–11 (from scratch)
Launches one `train_simclr` per GPU, each pinned via `CUDA_VISIBLE_DEVICES`,
training from random init to 50 epochs. Per-seed logs in `results/sweep_logs/`.
Idempotent + resumable — **re-run this cell to continue** after a session ends;
done seeds skip, partial seeds resume from their own `last_ckpt.pt`.

Check section 7's diagnostics after the first seeds land (new condition —
verify the cut before spending the rest of the quota).

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)

CONFIG = "configs/run/scale_strong.yaml"
SEEDS = list(range(12))            # 12 seeds/cell (H2 decidability; buffers gate failures)
N_GPUS = max(1, torch.cuda.device_count())

def run_id(seed): return f"scale_strong_seed{seed}"
def is_done(seed): return (ENC / run_id(seed) / "backbone.pt").exists()

def _launch(seed, gpu):
    rid = run_id(seed)
    env = dict(os.environ,
               CUDA_VISIBLE_DEVICES=str(gpu),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = open(LOGS / f"{rid}.log", "a")
    cmd = [sys.executable, "-m", "src.encoders.train_simclr", "--config", CONFIG,
           "--set", f"run.seed={seed}", "run.num_workers=2", "run.device=cuda"]
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
    return {"proc": p, "log": log, "rid": rid}

def run(n_gpus=N_GPUS, poll=15):
    subprocess.run(["pkill", "-9", "-f", "src.encoders.train_simclr"])  # reap orphans
    time.sleep(2)
    queue = [s for s in SEEDS if not is_done(s)]
    print(f"{len(SEEDS)} seeds | {len(SEEDS)-len(queue)} done | {len(queue)} to run on {n_gpus} GPU(s)")
    slots = [None] * n_gpus
    try:
        while queue or any(slots):
            for g in range(n_gpus):
                if slots[g] is None and queue:
                    seed = queue.pop(0)
                    if is_done(seed):
                        continue
                    slots[g] = _launch(seed, g)
                    print(f"[GPU{g}] start  {slots[g]['rid']}", flush=True)
            time.sleep(poll)
            for g in range(n_gpus):
                s = slots[g]
                if s and s["proc"].poll() is not None:
                    s["log"].close()
                    seed = int(s["rid"].split("seed")[1])
                    ok = is_done(seed)
                    print(f"[GPU{g}] {'DONE ' if ok else 'EXIT(rc=%s) ' % s['proc'].returncode}{s['rid']}", flush=True)
                    slots[g] = None
    except KeyboardInterrupt:
        for s in slots:
            if s: s["proc"].terminate()
        print("interrupted — checkpoints are saved; re-run this cell to resume")
        return
    print("done — all 12 seeds have a backbone.pt")

run()

## 6. Monitor
Re-run any time while section 5 is training.

In [ ]:
!nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv

## 7. Progress + final diagnostics
Per-seed state and, once done, the collapse diagnostics for each encoder.

In [ ]:
import json as _json

for seed in range(12):
    rid = f"scale_strong_seed{seed}"; d = ENC / rid
    if (d / "backbone.pt").exists():
        m = _json.loads((d / "metrics.json").read_text()); diag = m.get("diagnostics", {})
        print(f"{rid}: DONE loss={m['final_loss']:.4f} nan={m['nan_aborted']} epochs={m['epochs_run']} | "
              f"feat_std={diag.get('feat_std'):.4f} eff_rank={diag.get('eff_rank'):.1f} "
              f"align={diag.get('alignment'):.3f} unif={diag.get('uniformity'):.3f}")
    elif (d / "last_ckpt.pt").exists():
        last = None
        for line in (d / "train_log.jsonl").read_text().splitlines():
            try: last = _json.loads(line).get("epoch", last)
            except Exception: pass
        print(f"{rid}: partial @ep{last}")
    else:
        print(f"{rid}: todo")

## 8. Probe the cell → `stacks.npz` (the interface contract)
Fits the probe-capacity ladder over all 12 encoders + the 12 random-encoder
floor seeds and writes `results/probes/scale_strong/{stacks.npz, meta.json}`
— the only thing the H1–H4 statistics session depends on. The per-encoder
quality gate runs inside this step (gate-failed seeds are recorded in
`meta.json` and excluded downstream). Idempotent: skipped if the stack already
exists. Single-GPU is fine; expect **~3–4 h**.

In [ ]:
import subprocess, time, glob
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
TAG = "scale_strong"
N_SEEDS = 12
RANDOM_SEEDS = [str(i) for i in range(N_SEEDS)]  # seed-paired to trained for epsilon_G

if (REPO / "results" / "probes" / TAG / "stacks.npz").exists():
    print(f"{TAG}: stacks.npz already present -> nothing to do.")
else:
    encs = sorted(glob.glob(str(REPO / f"results/encoders/{TAG}_seed*/backbone.pt")))
    assert len(encs) == N_SEEDS, f"only {len(encs)}/{N_SEEDS} encoders trained -- finish section 5 first"
    cmd = ["python", "-m", "src.probes.run_sweep",
           "--config", "configs/probe/ladder.yaml",
           "--dataset", "shapes3d",
           "--condition", "scale", "--strength", "strong",
           "--encoders", *encs,
           "--random-seed", *RANDOM_SEEDS,
           "--epochs", "100", "--num-workers", "2"]
    t = time.time()
    rc = subprocess.run(cmd, cwd=str(REPO)).returncode
    print(f"{TAG}: {'OK' if rc == 0 else 'FAILED (exit %s)' % rc} in {(time.time()-t)/3600:.2f} h")

## 9. Save the finished cell
**Save Version (Commit)** to persist `/kaggle/working`. Deliverables: the 12
`backbone.pt` under `results/encoders/scale_strong_seed{0..11}/` and
`results/probes/scale_strong/{stacks.npz, meta.json}` — with these, the
scale arm is ready for the statistics layer.